# 002 — Deduplicate, split, preprocess

Runs `src/ml/preprocess.py` on the real data and checks the result. The logic lives in the module (the API reuses it); this notebook only inspects it.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import settings
from src.ml.preprocess import (
    TARGET, V_COLUMNS, build_preprocessor, deduplicate, load_raw,
    load_splits, save_splits, split, time_to_cyclical,
)

pd.set_option("display.float_format", "{:.4f}".format)

## 2. Load and remove duplicates

In [2]:
raw = load_raw()
df = deduplicate(raw)
print(f"{raw.shape} -> {df.shape}   removed {len(raw) - len(df)} duplicates")
print(f"fraud: {raw[TARGET].sum()} -> {df[TARGET].sum()}")

(284807, 31) -> (283726, 31)   removed 1081 duplicates
fraud: 492 -> 473


## 3. Stratified split + leakage checks

In [3]:
X_train, X_test, y_train, y_test = split(df)

summary = pd.DataFrame(
    {"rows": [len(df), len(X_train), len(X_test)],
     "fraud": [df[TARGET].sum(), y_train.sum(), y_test.sum()]},
    index=["full", "train", "test"],
)
summary["fraud_rate_%"] = summary["fraud"] / summary["rows"] * 100
print(summary)

print()
print("shared index labels:", len(X_train.index.intersection(X_test.index)))
print("identical rows across splits:", len(pd.merge(X_train, X_test, how="inner")))

         rows  fraud  fraud_rate_%
full   283726    473        0.1667
train  226980    378        0.1665
test    56746     95        0.1674

shared index labels: 0


identical rows across splits: 0


## 4. Fit on train only, transform both

In [4]:
preprocessor = build_preprocessor()
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(X_train_prep.shape, X_test_prep.shape)
print(list(preprocessor.get_feature_names_out()))
X_train_prep.head()

(226980, 31) (56746, 31)
['Amount', 'time_sin', 'time_cos', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28']


,Amount,time_sin,time_cos,V1,V2,V3,V4,V5,V6,V7,...,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28
225399,0.1449,-0.8853,-0.4650,2.2390,-1.7245,-2.1515,-2.5778,0.9937,3.5655,-1.7860,...,-0.3181,-0.3238,-0.1496,-0.0493,0.2784,0.6847,-0.2190,-0.1592,0.0379,-0.0499
133746,-0.4300,-0.4017,0.9158,-1.3151,1.6308,0.5970,-0.0384,-0.4046,-0.9657,0.2122,...,0.3924,-0.0676,-0.2389,-0.9468,0.3239,0.5156,-0.7130,-0.2665,-0.0178,0.0511
185792,-0.1498,0.1797,-0.9837,1.9088,0.0212,-2.0880,0.1293,1.1615,0.6052,-0.0224,...,-0.9947,-0.2105,0.2936,1.0958,-0.0449,-1.6895,0.1061,0.0078,0.0452,-0.0531
148925,-0.0941,0.3517,0.9361,1.8113,0.3166,0.3168,3.8802,0.0485,1.0202,-0.7349,...,-1.6935,-0.2280,0.1389,0.7004,0.1741,0.7030,-0.2125,-0.0100,-0.0177,-0.0380
18398,0.0324,0.8386,-0.5448,1.3588,-1.1209,0.5503,-1.5477,-1.1950,0.2754,-1.2018,...,-0.5918,-0.3617,-0.3410,-0.6364,0.2528,-0.3442,-0.0643,-0.4396,0.0625,0.0131


## 5. Inspect what was learned

In [5]:
scaler = preprocessor.named_transformers_["amount"].named_steps["scale"]
print(f"learned median of log1p(Amount): {scaler.center_[0]:.4f}  (= ${np.expm1(scaler.center_[0]):.2f})")
print(f"learned IQR    of log1p(Amount): {scaler.scale_[0]:.4f}")

assert not X_train_prep.isna().any().any() and not X_test_prep.isna().any().any()
assert np.allclose(X_test_prep[V_COLUMNS].to_numpy(), X_test[V_COLUMNS].to_numpy())
print("no NaNs; V1-V28 passed through unchanged")

pd.DataFrame({"train": X_train_prep["Amount"].describe(), "test": X_test_prep["Amount"].describe()})

learned median of log1p(Amount): 3.1390  (= $22.08)
learned IQR    of log1p(Amount): 2.4669
no NaNs; V1-V28 passed through unchanged


,train,test
count,226980.0000,56746.0000
mean,0.0070,0.0021
std,0.6714,0.6731
min,-1.2724,-1.2724
25%,-0.5020,-0.5143
50%,0.0000,-0.0088
75%,0.4980,0.4910
max,2.7351,2.8436


## 6. Why time is encoded on a circle

In [6]:
clock = pd.DataFrame({"Time": [23 * 3600 + 59 * 60, 60, 12 * 3600]}, index=["23:59", "00:01", "12:00"])
points = pd.DataFrame(time_to_cyclical(clock), index=clock.index, columns=["sin", "cos"])
print(points)
print("distance 23:59 <-> 00:01:", round(np.linalg.norm(points.loc["23:59"] - points.loc["00:01"]), 4))
print("distance 23:59 <-> 12:00:", round(np.linalg.norm(points.loc["23:59"] - points.loc["12:00"]), 4))

          sin     cos
23:59 -0.0044  1.0000
00:01  0.0044  1.0000
12:00 -0.0000 -1.0000
distance 23:59 <-> 00:01: 0.0087
distance 23:59 <-> 12:00: 2.0


## 7. Save and reload the splits

In [7]:
save_splits(X_train, X_test, y_train, y_test)
Xa, Xb, ya, yb = load_splits()
assert Xa.equals(X_train) and Xb.equals(X_test) and ya.equals(y_train) and yb.equals(y_test)
print("saved and reloaded identically:")
print(" ", settings.train_split_path)
print(" ", settings.test_split_path)

saved and reloaded identically:
  D:\Projects\fraud-detection-api\data\processed\train.parquet
  D:\Projects\fraud-detection-api\data\processed\test.parquet


## Step 1.2: what was done and why

- **Deduplicated before splitting.** 1,081 rows removed (19 fraud). Verified 0 identical rows across splits.
- **Stratified 80/20 split, seed 42.** Train 226,980 (378 fraud) / test 56,746 (95 fraud); fraud rate ~0.167% in both.
- **Preprocessor was fit on training data only.** Test data is only transformed, so there is no leakage from the scaler.
- **Amount:** log1p + RobustScaler (median/IQR ignore the extreme amounts that fraud produces).
- **Time:** converted to time-of-day sin/cos. Raw "seconds since dataset start" doesn't exist for a live transaction.
- **V1–V28:** passed through unchanged (PCA output is already centered).
- **Only 95 test fraud cases:** each one is ~1% of recall. Treat metric differences under a few % as noise.